In [1]:
!pip install ipywidgets

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.2 MB 3.7 MB/s eta 0:00:01
   ----------------------- ---------------- 1.3/2.2 MB 3.9 MB/s eta 0:00:01
   --------------------------------- ------ 1.8/2.2 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 3.0 MB/s  0:00:00

   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   --------

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
import datetime
import json

In [4]:
# Load both datasets
df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\dropped_new_data.csv")

In [ ]:
df["payment_type"]

Index(['payment_type', 'profit_per_order', 'sales_per_customer',
       'category_name', 'customer_segment', 'department_name', 'latitude',
       'longitude', 'market', 'order_city', 'order_country', 'order_date',
       'order_item_discount', 'order_item_discount_rate',
       'order_item_product_price', 'order_item_profit_ratio',
       'order_item_quantity', 'sales', 'order_item_total_amount',
       'order_profit_per_order', 'order_region', 'order_state', 'order_status',
       'product_name', 'shipping_date', 'shipping_mode', 'label',
       'dest_latitude', 'dest_longitude', 'store_to_order_distance_km',
       'distance_normalized', 'customer_country', 'is_international',
       'customer_city', 'customer_state'],
      dtype='object')

In [12]:
# -----------------------------
# 1. Customer Location Hierarchy
# -----------------------------
customer_hierarchy = {}

for _, row in df.iterrows():
    country = row['customer_country']
    state = row['customer_state']
    city = row['customer_city']
    
    customer_hierarchy.setdefault(country, {})
    customer_hierarchy[country].setdefault(state, [])
    
    if city not in customer_hierarchy[country][state]:
        customer_hierarchy[country][state].append(city)

# Save as JSON
with open("customer_hierarchy.json", "w", encoding="utf-8") as f:
    json.dump(customer_hierarchy, f, indent=4, ensure_ascii=False)



# -----------------------------
# 2. Order Location Hierarchy (Store + Destination)
# -----------------------------
order_hierarchy = {}

for _, row in df.iterrows():
    region = row['order_region']
    country = row['order_country']
    state = row['order_state']
    city = row['order_city']

    # store (origin) lat/lon
    store_lat, store_lon = row['latitude'], row['longitude']

    # destination (customer) lat/lon
    dest_lat, dest_lon = row['dest_latitude'], row['dest_longitude']

    order_hierarchy.setdefault(region, {})
    order_hierarchy[region].setdefault(country, {})
    order_hierarchy[region][country].setdefault(state, {})

    order_hierarchy[region][country][state][city] = {
        "store_location": {
            "latitude": store_lat,
            "longitude": store_lon
        },
        "customer_location": {
            "latitude": dest_lat,
            "longitude": dest_lon
        }
    }

# Save as JSON
with open("order_hierarchy.json", "w", encoding="utf-8") as f:
    json.dump(order_hierarchy, f, indent=4, ensure_ascii=False)

In [ ]:
import json
import ipywidgets as widgets
from IPython.display import display
import datetime

# --------------------------
# Load Hierarchy JSON Files
# --------------------------
with open("customer_hierarchy.json", "r", encoding="utf-8") as f:
    customer_hierarchy = json.load(f)

with open("order_hierarchy.json", "r", encoding="utf-8") as f:
    order_hierarchy = json.load(f)

# --------------------------
# Example other inputs
# --------------------------
payment_types = ["CASH", "DEBIT", "PAYMENT", "TRANSFER"]
shipping_modes = ["Standard Class", "First Class", "Second Class"]

# Widgets
profit_input = widgets.FloatText(description="Profit per order")
price_input = widgets.FloatText(description="Product price")
quantity_input = widgets.IntText(description="Quantity")
discount_input = widgets.FloatSlider(min=0, max=1, step=0.01, description="Discount %")

# Customer Location Dropdowns
customer_country_dropdown = widgets.Dropdown(
    options=list(customer_hierarchy.keys()), 
    description="Customer Country"
)
customer_state_dropdown = widgets.Dropdown(description="Customer State")
customer_city_dropdown = widgets.Dropdown(description="Customer City")

# Order Location Dropdowns
order_region_dropdown = widgets.Dropdown(
    options=list(order_hierarchy.keys()), 
    description="Order Region"
)
order_country_dropdown = widgets.Dropdown(description="Order Country")
order_state_dropdown = widgets.Dropdown(description="Order State")
order_city_dropdown = widgets.Dropdown(description="Order City")

# Payment + Shipping
payment_dropdown = widgets.Dropdown(options=payment_types, description="Payment Type")
shipping_dropdown = widgets.Dropdown(options=shipping_modes, description="Shipping Mode")

# --------------------------
# Date Pickers (No timezone)
# --------------------------
order_date = widgets.DatePicker(
    description="Order Date",
    value=datetime.date.today()
)

ship_date = widgets.DatePicker(
    description="Ship Date",
    value=datetime.date.today() + datetime.timedelta(days=2)
)

# --------------------------
# Validation: Ship date > Order date
# --------------------------
def validate_dates(change):
    if order_date.value and ship_date.value:
        if ship_date.value <= order_date.value:
            print("❌ Ship Date must be later than Order Date")
        else:
            print("✅ Dates are valid")

order_date.observe(validate_dates, names="value")
ship_date.observe(validate_dates, names="value")


# --------------------------
# Dynamic Updates
# --------------------------

# Customer state updates
def update_customer_states(change):
    states = list(customer_hierarchy[change["new"]].keys())
    customer_state_dropdown.options = states
    customer_state_dropdown.value = states[0] if states else None

update_customer_states({"new": customer_country_dropdown.value})
customer_country_dropdown.observe(update_customer_states, names="value")

# Customer city updates
def update_customer_cities(change):
    cities = customer_hierarchy[customer_country_dropdown.value][change["new"]]
    customer_city_dropdown.options = cities
    customer_city_dropdown.value = cities[0] if cities else None

update_customer_cities({"new": customer_state_dropdown.value})
customer_state_dropdown.observe(update_customer_cities, names="value")


# Order country updates
def update_order_countries(change):
    countries = list(order_hierarchy[change["new"]].keys())
    order_country_dropdown.options = countries
    order_country_dropdown.value = countries[0] if countries else None

update_order_countries({"new": order_region_dropdown.value})
order_region_dropdown.observe(update_order_countries, names="value")

# Order state updates
def update_order_states(change):
    states = list(order_hierarchy[order_region_dropdown.value][change["new"]].keys())
    order_state_dropdown.options = states
    order_state_dropdown.value = states[0] if states else None

update_order_states({"new": order_country_dropdown.value})
order_country_dropdown.observe(update_order_states, names="value")

# Order city updates
def update_order_cities(change):
    cities = list(order_hierarchy[order_region_dropdown.value][order_country_dropdown.value][change["new"]].keys())
    order_city_dropdown.options = cities
    order_city_dropdown.value = cities[0] if cities else None

update_order_cities({"new": order_state_dropdown.value})
order_state_dropdown.observe(update_order_cities, names="value")


# --------------------------
# Display Widgets
# --------------------------
display(
    profit_input, price_input, quantity_input, discount_input,
    customer_country_dropdown, customer_state_dropdown, customer_city_dropdown,
    order_region_dropdown, order_country_dropdown, order_state_dropdown, order_city_dropdown,
    payment_dropdown, shipping_dropdown, order_date, ship_date
)
